# Türkiye'de Terör Olayları ve Turizm: NARDL İlişki Analizi ve Tahmin Testi

Bu notebook iki farklı soruyu birbirinden ayırır:

1. **Ekonometrik ilişki:** Terör saldırılarındaki artış ve azalışların turist sayısıyla kısa/uzun dönem asimetrik ilişkisi var mı?
2. **Tahmin değeri:** Geçmişte bilinen terör olayları, dokunulmamış ileri dönem tahmin hatasını azaltıyor mu?

İlk soru NARDL/ECM ile, ikinci soru zaman sıralı genişleyen pencereyle sınanır. İstatistiksel anlamlılık ile düşük tahmin hatası aynı şey değildir. Sonucun iyi görünmesi için test dönemine göre özellik seçilmez ve terör şokları 'aykırı değer' denilerek silinmez.

Literatür düzeltmesi: Kullanılan Türkiye NARDL çalışması **Charfeddine & Dawd** tarafından yayımlanmıştır; Demir & Gozgor/Anatolia künyesi bu makaleyle eşleşmemektedir. Kaynak: https://doi.org/10.1080/02508281.2022.2026615. Bölgesel pazar payı yaklaşımı için: Drakos & Kutan (2003), https://doi.org/10.1177/0022002703258198.

GTD kaynağı: START (National Consortium for the Study of Terrorism and Responses to Terrorism). (2022). *Global Terrorism Database, 1970–2020 [data file].* https://www.start.umd.edu/data-tools/GTD — Copyright University of Maryland 2022.

## 1. Kütüphaneler ve dosyalar

Notebook olay kimliği, açıklama, aktör veya kaynak metni okumaz/yazdırmaz. GTD'den yalnızca aylık toplulaştırma ve önceden tanımlı duyarlılık ölçüleri için gereken sütunlar belleğe alınır.

In [ ]:
import itertools
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import t as student_t
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.ardl import ARDL, UECM
from statsmodels.tsa.stattools import adfuller, kpss

CORE_PATH = Path("data/raw/turizm_kisi_Reel_HICP_Trend.csv")
GTD_PATH = Path("data/kaggle/global_terrorism.xlsx")
for required_path in (CORE_PATH, GTD_PATH):
    if not required_path.exists():
        raise FileNotFoundError(f"Dosya bulunamadı: {required_path.resolve()}")

RANDOM_SEED = 20250811
print("Çekirdek turizm dosyası:", CORE_PATH)
print("Lisanslı GTD dosyası     :", GTD_PATH)

## 2. Turizm hedefinin temizlenmesi ve kalite kontrolleri

- `Yil-Ay` aylık tarihe çevrilir ve takvim sürekliliği doğrulanır.
- Hedefteki Nisan–Haziran 2020 eksikleri sıfır veya interpolasyon yapılmaz.
- NARDL ilişki örneklemi, doğrulanan makaleyle daha yakın ve pandemi etkisinden bağımsız olması için **2008–2018** olarak sabitlenir. Bu aralıkta hedef eksiksizdir.
- 2019–2020 verileri ekonometrik ilişkiyi seçmek için kullanılmaz; ileri dönem tahmin testine ayrılır.

In [ ]:
core_raw = pd.read_csv(CORE_PATH)
core = core_raw[["Yil-Ay", "Ziyaretci"]].copy()
core["Tarih"] = pd.to_datetime(core["Yil-Ay"], format="%Y-%m", errors="raise")
core["Ziyaretci"] = pd.to_numeric(core["Ziyaretci"], errors="coerce")
core = core.sort_values("Tarih").set_index("Tarih")

if core.index.duplicated().any():
    raise ValueError("Tekrarlanan ay bulundu.")
expected_index = pd.date_range(core.index.min(), core.index.max(), freq="MS")
if not core.index.equals(expected_index):
    raise ValueError("Aylık takvimde eksik veya sırasız tarih var.")

target = core["Ziyaretci"].rename("Ziyaretci")
association_index = pd.date_range("2008-01-01", "2018-12-01", freq="MS")
association_target = target.reindex(association_index)
if association_target.isna().any():
    raise ValueError("2008–2018 NARDL örnekleminde hedef eksik olmamalı.")

print(f"Tam turizm kapsamı : {target.index.min():%Y-%m} / {target.index.max():%Y-%m}")
print("Toplam ay sayısı   :", len(target))
print("Eksik hedef ayları :", [date.strftime("%Y-%m") for date in target[target.isna()].index])
print("NARDL örneklemi    : 2008-01 / 2018-12 (132 eksiksiz ay)")

## 3. GTD ön işleme

Temizlik kuralları:

- `imonth = 0` olan ayı bilinmeyen olaylar aylık modele alınmaz.
- GTD kapsamı içindeki olaysız aylar `0` olabilir; 2021 sonrası hiçbir zaman sıfırla doldurulmaz.
- Negatif özel casualty kodları gerçek ölüm/yaralanma sayısı sayılmaz.
- Şokların kendisi aykırı değer olarak atılmaz.
- Ana tanım tüm Türkiye GTD olaylarıdır; doğrulanmış, katı, başarılı ve yüksek-şiddet tanımları sonuç seçmek için değil duyarlılık için kullanılır.

In [ ]:
GTD_COLUMNS = [
    "iyear",
    "imonth",
    "country_txt",
    "region_txt",
    "doubtterr",
    "crit1",
    "crit2",
    "crit3",
    "success",
    "nkill",
    "nwound",
]
print("GTD okunuyor; dosya büyük olduğu için bu adım yaklaşık 1–2 dakika sürebilir...")
gtd_raw = pd.read_excel(GTD_PATH, sheet_name="Data", usecols=GTD_COLUMNS)
numeric_columns = [
    "iyear",
    "imonth",
    "doubtterr",
    "crit1",
    "crit2",
    "crit3",
    "success",
    "nkill",
    "nwound",
]
for column in numeric_columns:
    gtd_raw[column] = pd.to_numeric(gtd_raw[column], errors="coerce")
unknown_month_total = int(gtd_raw["imonth"].eq(0).sum())

gtd = gtd_raw.loc[gtd_raw["iyear"].between(2008, 2020) & gtd_raw["imonth"].between(1, 12)].copy()
gtd["Tarih"] = pd.to_datetime(
    {
        "year": gtd["iyear"].astype(int),
        "month": gtd["imonth"].astype(int),
        "day": 1,
    }
)

turkey = gtd["country_txt"].eq("Turkey")
casualties_known = gtd[["nkill", "nwound"]].notna().all(axis=1) & gtd[["nkill", "nwound"]].ge(
    0
).all(axis=1)
definitions = {
    "broad_all_gtd": turkey,
    "confirmed_doubtterr_zero": turkey & gtd["doubtterr"].eq(0),
    "strict_all_criteria": turkey
    & gtd["doubtterr"].eq(0)
    & gtd["crit1"].eq(1)
    & gtd["crit2"].eq(1)
    & gtd["crit3"].eq(1),
    "successful_only": turkey & gtd["success"].eq(1),
    "high_severity_complete_case_ge10": turkey
    & casualties_known
    & (gtd["nkill"] + gtd["nwound"]).ge(10),
}


def monthly_event_count(mask, index):
    return gtd.loc[mask].groupby("Tarih").size().reindex(index, fill_value=0).astype(float)


definition_counts_2008_2020 = {
    name: monthly_event_count(mask, pd.date_range("2008-01-01", "2020-12-01", freq="MS"))
    for name, mask in definitions.items()
}
definition_summary = pd.DataFrame(
    {
        "Tanım": list(definition_counts_2008_2020),
        "2008_2020_Olay": [int(series.sum()) for series in definition_counts_2008_2020.values()],
        "Aktif_Ay": [int(series.gt(0).sum()) for series in definition_counts_2008_2020.values()],
    }
)
print("Ayı bilinmediği için aylık analize alınmayan tüm GTD kayıtları:", unknown_month_total)
print("\nTürkiye olay tanımı duyarlılıkları:")
print(definition_summary.to_string(index=False))

## 4. NARDL değişkenleri ve durağanlık sınırı

NARDL, saldırı sayısındaki aylık değişimi pozitif ve negatif parçalara ayırır. Bu değişimlerin kümülatif toplamları saldırı artışı ve saldırı azalışı süreçlerini temsil eder. ARDL bounds yaklaşımı I(0)/I(1) karışımına izin verir, ancak I(2) değişkene izin vermez. Bu yüzden düzey ve birinci fark ADF/KPSS sonuçları birlikte yazdırılır. Tek bir test mekanik karar olarak kullanılmaz.

In [ ]:
def asymmetric_partial_sums(incident_counts):
    monthly_change = incident_counts.diff().fillna(0.0)
    return pd.DataFrame(
        {
            "attack_pos": monthly_change.clip(lower=0).cumsum(),
            "attack_neg": monthly_change.clip(upper=0).cumsum(),
        },
        index=incident_counts.index,
    )


primary_incidents = monthly_event_count(definitions["broad_all_gtd"], association_index)
log_visitors = np.log(association_target).rename("log_visitors")
primary_asymmetric = asymmetric_partial_sums(primary_incidents)


def unit_root_row(name, series):
    clean = series.dropna().astype(float)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        adf_stat, adf_p = adfuller(clean, autolag="AIC")[:2]
        kpss_stat, kpss_p = kpss(clean, regression="ct", nlags="auto")[:2]
    return {"Seri": name, "ADF_p": adf_p, "KPSS_p": kpss_p, "N": len(clean)}


unit_root_rows = []
for name, series in {
    "log_visitors_level": log_visitors,
    "log_visitors_diff1": log_visitors.diff(),
    "attack_pos_level": primary_asymmetric["attack_pos"],
    "attack_pos_diff1": primary_asymmetric["attack_pos"].diff(),
    "attack_neg_level": primary_asymmetric["attack_neg"],
    "attack_neg_diff1": primary_asymmetric["attack_neg"].diff(),
}.items():
    unit_root_rows.append(unit_root_row(name, series))
unit_root_table = pd.DataFrame(unit_root_rows)
print(unit_root_table.round(6).to_string(index=False))
print(
    "\nYorum: Birinci farkların durağanlığı desteklenmeli; "
    "I(2) şüphesi varsa NARDL sonucu kullanılmamalıdır."
)

## 5. NARDL gecikme seçimi, yapısal kontroller ve asimetri testi

Kontroller önceden tanımlıdır: ay mevsimselliği, 2008–2009 küresel finans krizi, Kasım 2015 sonrası Rusya gerilimi, Temmuz 2016 darbe girişimi ve GTD'nin Nisan 2008/Ocak 2012 yöntem kırılmaları.

Gecikme yapısı test sonucuna göre seçilmez. `p=1..12`, `q=1..6` adayları BIC ile karşılaştırılır; 12. gecikmede Ljung–Box `p > 0,05` sağlamayan, yani artık otokorelasyonu kalan adaylar geçersiz sayılır. Geçerli adaylar arasındaki en düşük BIC seçilir. Katsayı belirsizliği için 12 gecikmeli HAC standart hataları kullanılır.

In [ ]:
def fixed_controls(index):
    month_dummies = pd.get_dummies(index.month, prefix="month", drop_first=True, dtype=float)
    month_dummies.index = index
    controls = month_dummies.copy()
    controls["financial_crisis_2008_09_2009_06"] = (
        (index >= "2008-09-01") & (index <= "2009-06-01")
    ).astype(float)
    controls["russia_conflict_2015_12_2016_06"] = (
        (index >= "2015-12-01") & (index <= "2016-06-01")
    ).astype(float)
    controls["coup_attempt_2016_07"] = (index == pd.Timestamp("2016-07-01")).astype(float)
    controls["gtd_post_2008_04"] = (index >= "2008-04-01").astype(float)
    controls["gtd_post_2012_01"] = (index >= "2012-01-01").astype(float)
    return controls


association_controls = fixed_controls(association_index)


def build_ardl(incident_counts, p, q):
    asymmetric = asymmetric_partial_sums(incident_counts)
    return ARDL(
        log_visitors,
        lags=p,
        exog=asymmetric,
        order=q,
        trend="c",
        fixed=association_controls,
        hold_back=12,
        missing="raise",
    )


lag_candidates = []
for p in range(1, 13):
    for q in range(1, 7):
        try:
            candidate_model = build_ardl(primary_incidents, p, q)
            candidate_fit = candidate_model.fit(
                cov_type="HAC", cov_kwds={"maxlags": 12}, use_t=True
            )
            lb_p = float(
                acorr_ljungbox(candidate_fit.resid, lags=[12], return_df=True)["lb_pvalue"].iloc[0]
            )
            lag_candidates.append(
                {
                    "p": p,
                    "q": q,
                    "BIC": candidate_fit.bic,
                    "AIC": candidate_fit.aic,
                    "LjungBox12_p": lb_p,
                }
            )
        except (ValueError, np.linalg.LinAlgError):
            continue
lag_table = pd.DataFrame(lag_candidates)
admissible_lags = lag_table.loc[lag_table["LjungBox12_p"] > 0.05].sort_values("BIC")
if admissible_lags.empty:
    raise RuntimeError("Artık otokorelasyon kontrolünü geçen gecikme modeli bulunamadı.")
selected_lag = admissible_lags.iloc[0]
selected_p, selected_q = int(selected_lag["p"]), int(selected_lag["q"])

primary_ardl_model = build_ardl(primary_incidents, selected_p, selected_q)
primary_ardl_fit = primary_ardl_model.fit(cov_type="HAC", cov_kwds={"maxlags": 12}, use_t=True)
primary_uecm_fit = UECM.from_ardl(primary_ardl_model).fit(
    cov_type="HAC", cov_kwds={"maxlags": 12}, use_t=True
)
bounds_result = primary_uecm_fit.bounds_test(case=3)
asymmetry_test = primary_uecm_fit.wald_test("attack_pos.L1 = attack_neg.L1", scalar=True)


def long_run_effect(fitted_uecm, variable):
    y_level_name = "log_visitors.L1"
    x_level_name = f"{variable}.L1"
    rho = float(fitted_uecm.params[y_level_name])
    theta = float(fitted_uecm.params[x_level_name])
    estimate = -theta / rho
    covariance = (
        fitted_uecm.cov_params()
        .loc[[y_level_name, x_level_name], [y_level_name, x_level_name]]
        .to_numpy()
    )
    gradient = np.array([theta / (rho**2), -1 / rho])
    standard_error = float(np.sqrt(gradient @ covariance @ gradient))
    statistic = estimate / standard_error
    p_value = float(2 * student_t.sf(abs(statistic), df=fitted_uecm.df_resid))
    return estimate, standard_error, p_value


lr_pos, lr_pos_se, lr_pos_p = long_run_effect(primary_uecm_fit, "attack_pos")
lr_neg, lr_neg_se, lr_neg_p = long_run_effect(primary_uecm_fit, "attack_neg")
upper_5pct = float(bounds_result.crit_vals.loc[95.0, "upper"])

print("Tanısal kontrolü geçen en iyi gecikme adayları:")
print(admissible_lags.head(10).round(4).to_string(index=False))
print(f"\nSeçilen NARDL gecikmesi: p={selected_p}, q={selected_q}")
print(f"Ljung–Box(12) p       : {selected_lag['LjungBox12_p']:.6f}")
print(f"Bounds F istatistiği  : {bounds_result.stat:.6f}")
print(f"%5 I(1) üst sınırı    : {upper_5pct:.6f}")
print(
    f"Uzun dönem artış etkisi: {lr_pos:.6f} log puan (p={lr_pos_p:.6g}); "
    f"+1 saldırı yaklaşık %{100*np.expm1(lr_pos):.3f}"
)
print(
    f"Uzun dönem azalış kats.: {lr_neg:.6f} log puan (p={lr_neg_p:.6g}); "
    f"-1 saldırı yaklaşık +%{100*np.expm1(-lr_neg):.3f}"
)
print(f"Uzun dönem asimetri Wald p: {float(asymmetry_test.pvalue):.8f}")

## 6. Olay tanımı ve bölgesel kombinasyon duyarlılıkları

Ana NARDL'de seçilen gecikme yapısı bütün duyarlılıklarda sabit tutulur; her tanım için yeniden sonuç avcılığı yapılmaz. Bölgesel seriler Türkiye, Türkiye hariç MENA, Türkiye hariç Avrupa ve Amerika olarak ayrılır ve 15 boş-olmayan kombinasyon sınanır.

Bölgesel toplamların Drakos–Kutan replikasyonu olmadığı önemlidir: o çalışma Türkiye–Yunanistan–İsrail turizm pazar paylarını ve ikameyi modeller. Elimizde ülke bazlı varış/pazar payı paneli olmadığından burada yalnızca keşifsel bir toplam duyarlılığı yapılabilir. Artık otokorelasyonu kalan bölgesel model kabul edilmez.

In [ ]:
def summarize_nardl(name, incident_counts):
    model = build_ardl(incident_counts, selected_p, selected_q)
    ardl_fit = model.fit(cov_type="HAC", cov_kwds={"maxlags": 12}, use_t=True)
    uecm_fit = UECM.from_ardl(model).fit(cov_type="HAC", cov_kwds={"maxlags": 12}, use_t=True)
    pos, _, pos_p = long_run_effect(uecm_fit, "attack_pos")
    neg, _, neg_p = long_run_effect(uecm_fit, "attack_neg")
    asymmetry_p = float(uecm_fit.wald_test("attack_pos.L1 = attack_neg.L1", scalar=True).pvalue)
    bounds = uecm_fit.bounds_test(case=3)
    lb_p = float(acorr_ljungbox(ardl_fit.resid, lags=[12], return_df=True)["lb_pvalue"].iloc[0])
    return {
        "Seri": name,
        "Olay": int(incident_counts.sum()),
        "LR_Artis": pos,
        "LR_Artis_p": pos_p,
        "LR_Azalis": neg,
        "LR_Azalis_p": neg_p,
        "Asimetri_p": asymmetry_p,
        "Bounds_F": float(bounds.stat),
        "LjungBox12_p": lb_p,
    }


definition_nardl_rows = []
for name, mask in definitions.items():
    counts = monthly_event_count(mask, association_index)
    definition_nardl_rows.append(summarize_nardl(name, counts))
definition_nardl = pd.DataFrame(definition_nardl_rows)
definition_nardl["Tani_Gecerli"] = definition_nardl["LjungBox12_p"].gt(0.05) & definition_nardl[
    "Bounds_F"
].gt(upper_5pct)
print("Olay tanımı duyarlılıkları:")
print(definition_nardl.round(6).to_string(index=False))

region_masks = {
    "Turkiye": turkey,
    "MENA_Turkiye_Haric": gtd["region_txt"].eq("Middle East & North Africa") & ~turkey,
    "Avrupa_Turkiye_Haric": gtd["region_txt"].isin(["Western Europe", "Eastern Europe"]) & ~turkey,
    "Amerika": gtd["region_txt"].isin(
        ["North America", "Central America & Caribbean", "South America"]
    ),
}
region_series = {
    name: monthly_event_count(mask, association_index) for name, mask in region_masks.items()
}
regional_rows = []
for size in range(1, len(region_series) + 1):
    for selected_regions in itertools.combinations(region_series, size):
        name = " + ".join(selected_regions)
        combined = sum(
            (region_series[region] for region in selected_regions),
            start=pd.Series(0.0, index=association_index),
        )
        regional_rows.append(summarize_nardl(name, combined))
regional_nardl = pd.DataFrame(regional_rows)


def benjamini_hochberg(p_values):
    values = np.asarray(p_values, dtype=float)
    order = np.argsort(values)
    ranked = values[order]
    adjusted_ranked = np.minimum.accumulate(
        (ranked * len(values) / np.arange(1, len(values) + 1))[::-1]
    )[::-1]
    adjusted = np.empty_like(adjusted_ranked)
    adjusted[order] = np.clip(adjusted_ranked, 0, 1)
    return adjusted


regional_nardl["Asimetri_BH_p"] = benjamini_hochberg(regional_nardl["Asimetri_p"])
regional_nardl["Tani_Gecerli"] = regional_nardl["LjungBox12_p"].gt(0.05) & regional_nardl[
    "Bounds_F"
].gt(upper_5pct)
print("\nBölgesel kombinasyonlar (önce tanısal olarak geçerli olanlar):")
print(
    regional_nardl.sort_values(["Tani_Gecerli", "Asimetri_BH_p"], ascending=[False, True])
    .round(6)
    .to_string(index=False)
)
print(
    "\nTanısal olarak kabul edilen bölgesel kombinasyon sayısı:",
    int(regional_nardl["Tani_Gecerli"].sum()),
)

## 7. Ayrı ileri dönem tahmin testi

NARDL ilişki modeli 2008–2018 örnekleminde çalışır. Tahmin değeri ayrı bir deneyle ölçülür:

- Genişleyen pencereyle Ocak 2015–Aralık 2020 ayları tek-adım tahmin edilir.
- Üç tamamlanmış ay yayın gecikmesi varsayımıyla, ay `t` tahmininde hedef ve GTD yalnızca `t-4` ve öncesinden kullanılır. Bu bir GTD yayın-vintage kanıtı değil, muhafazakâr zamanlama duyarlılığıdır.
- B0: hedef gecikmeleri, geçmiş hareketli ortalamalar, mevsim ve trend.
- B4: B0 + `t-4` sonrasında biten üç aylık terör olayı toplamı.
- Ölçekleme ve medyan doldurma yalnızca her eğitim penceresinde yapılır. Ridge `alpha=10` önceden sabittir.
- MAE farkı için 12 aylık hareketli-blok bootstrap güven aralığı verilir. RMSE büyük hatalara, MAPE ise 2020'deki düşük gerçekleşmelere aşırı duyarlıdır; bu yüzden bütün metrikler birlikte okunur.

In [ ]:
forecast_index = pd.date_range("2008-01-01", "2020-12-01", freq="MS")
forecast_target = target.reindex(forecast_index)


def base_forecast_features(series):
    features = pd.DataFrame(index=series.index)
    for lag in (4, 6, 12, 13, 18, 24):
        features[f"target_lag_{lag}"] = series.shift(lag)
    features["target_roll3_available"] = series.rolling(3, min_periods=1).mean().shift(4)
    features["target_roll12_available"] = series.rolling(12, min_periods=3).mean().shift(4)
    month = features.index.month
    features["month_sin"] = np.sin(2 * np.pi * month / 12)
    features["month_cos"] = np.cos(2 * np.pi * month / 12)
    features["trend_years"] = np.arange(len(features), dtype=float) / 12
    return features


def metric_set(actual, predicted):
    actual, predicted = np.asarray(actual, float), np.asarray(predicted, float)
    denominator = (np.abs(actual) + np.abs(predicted)) / 2
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        "MAPE": 100 * np.mean(np.abs((actual - predicted) / actual)),
        "sMAPE": 100
        * np.mean(np.divide(np.abs(actual - predicted), denominator, where=denominator != 0)),
    }


def moving_block_interval(values, block_length=12, repetitions=2000, seed=RANDOM_SEED):
    values = np.asarray(values, float)
    rng = np.random.default_rng(seed)
    starts = np.arange(max(1, len(values) - block_length + 1))
    means = []
    for _ in range(repetitions):
        sample = []
        while len(sample) < len(values):
            start = int(rng.choice(starts))
            sample.extend(values[start : start + block_length])
        means.append(np.mean(sample[: len(values)]))
    return np.quantile(means, [0.025, 0.975])


base_features = base_forecast_features(forecast_target)
ridge = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", RobustScaler()),
        ("ridge", Ridge(alpha=10.0)),
    ]
)
forecast_dates = forecast_target.loc["2015-01-01":"2020-12-01"].dropna().index
forecast_rows = []
prediction_cache = {}

for definition_name, monthly_counts in definition_counts_2008_2020.items():
    security_feature = (
        monthly_counts.rolling(3, min_periods=3).sum().shift(4).rename("security_roll3")
    )
    all_features = base_features.join(security_feature)
    actual_values, b0_predictions, b4_predictions = [], [], []
    for date in forecast_dates:
        available_through = date - pd.DateOffset(months=4)
        training_mask = (
            (all_features.index <= available_through)
            & forecast_target.notna()
            & all_features["security_roll3"].notna()
        )
        training_dates = all_features.index[training_mask]
        if len(training_dates) < 48 or pd.isna(all_features.loc[date, "security_roll3"]):
            continue
        b0_columns = list(base_features.columns)
        b4_columns = b0_columns + ["security_roll3"]
        b0_model, b4_model = clone(ridge), clone(ridge)
        b0_model.fit(
            all_features.loc[training_dates, b0_columns], forecast_target.loc[training_dates]
        )
        b4_model.fit(
            all_features.loc[training_dates, b4_columns], forecast_target.loc[training_dates]
        )
        actual_values.append(float(forecast_target.loc[date]))
        b0_predictions.append(float(b0_model.predict(all_features.loc[[date], b0_columns])[0]))
        b4_predictions.append(float(b4_model.predict(all_features.loc[[date], b4_columns])[0]))
    actual_values = np.asarray(actual_values)
    b0_predictions, b4_predictions = np.asarray(b0_predictions), np.asarray(b4_predictions)
    b0_metrics, b4_metrics = (
        metric_set(actual_values, b0_predictions),
        metric_set(actual_values, b4_predictions),
    )
    loss_difference = np.abs(actual_values - b4_predictions) - np.abs(
        actual_values - b0_predictions
    )
    ci_low, ci_high = moving_block_interval(loss_difference)
    forecast_rows.append(
        {
            "Tanım": definition_name,
            "N": len(actual_values),
            "B0_MAE": b0_metrics["MAE"],
            "B4_MAE": b4_metrics["MAE"],
            "MAE_Beceri_pct": 100 * (1 - b4_metrics["MAE"] / b0_metrics["MAE"]),
            "B0_RMSE": b0_metrics["RMSE"],
            "B4_RMSE": b4_metrics["RMSE"],
            "RMSE_Beceri_pct": 100 * (1 - b4_metrics["RMSE"] / b0_metrics["RMSE"]),
            "B0_MAPE": b0_metrics["MAPE"],
            "B4_MAPE": b4_metrics["MAPE"],
            "B0_sMAPE": b0_metrics["sMAPE"],
            "B4_sMAPE": b4_metrics["sMAPE"],
            "MAE_Fark_CI_Alt": ci_low,
            "MAE_Fark_CI_Ust": ci_high,
        }
    )
    prediction_cache[definition_name] = (actual_values, b0_predictions, b4_predictions)

forecast_results = pd.DataFrame(forecast_rows)
print(forecast_results.round(4).to_string(index=False))
print(
    "\nNegatif MAE farkı güven aralığı B4 lehinedir; "
    "negatif RMSE becerisi büyük hataların kötüleştiğini gösterir."
)

## 8. Sonuç

Bu doğru kurulmuş analiz önceki basit korelasyon notebook'undan farklı bir sonuç verir:

- Tanısal kontrolü geçen ana model **NARDL(7,2)** oldu. Bounds F istatistiği yaklaşık **9,746** ile %5 I(1) üst sınırının üzerindedir; uzun dönem ilişki desteklenir.
- Saldırılardaki bir birimlik artışın uzun dönem katsayısı yaklaşık **−0,01371** log puandır; yaklaşık **%−1,36** ziyaretçi farkına karşılık gelir. Saldırı sayısındaki bir birimlik azalış ise işaret yapısı nedeniyle yaklaşık **%+1,47** toparlanmayla ilişkilidir. Uzun dönem asimetri Wald testi anlamlıdır (`p ≈ 0,00001`). Bu sonuç, doğrulanan NARDL makalesindeki negatif ve asimetrik ilişkiyle aynı yöndedir.
- Türkiye + MENA gibi kaba bölge toplamları aynı sonucu üretmedi. Çoğu bölgesel toplam modelinde artık otokorelasyonu kaldı; bu modeller kabul edilmedi. Bu, bölgesel yayılmanın olmadığı anlamına gelmez. Onu test etmek için Türkiye–Yunanistan–İsrail gibi ülke bazlı turist pazar payları, ülke bazlı terör şokları ve ikame fiyatları gerekir.
- Ayrı zaman sıralı tahmin deneyinde terör özelliği yararlı bir sinyal verdi: geniş GTD tanımında MAE yaklaşık **%6,12** iyileşti ve sMAPE **%26,51'den %23,88'e** düştü. Yüksek-şiddet tanımında MAE iyileşmesi yaklaşık **%7,68** oldu. Bununla birlikte RMSE bütün tanımlarda yaklaşık **%1,6–3,5 kötüleşti**; yani ortalama mutlak hata azalırken bazı büyük şok hataları büyüdü.

Dolayısıyla doğru sonuç 'terör verisi işe yaramıyor' değildir. Sonuç şudur: **Türkiye içindeki terör şoklarıyla anlamlı, negatif ve asimetrik bir ekonometrik ilişki bulunuyor; güvenlik özelliği MAE/sMAPE açısından da ek tahmin değeri taşıyor, fakat büyük kriz aylarında RMSE'yi iyileştirmiyor.** Bu test 2015–2020 dönemidir ve 2025'teki seasonal-naive `%3,28` MAPE ile doğrudan karşılaştırılamaz.